In [22]:

from pathlib import Path
import pandas as pd
import numpy as np

RAW_PATH = Path("/home/bcrlab/igguest/porat_naama/data/raw/geographic-sum-per-day-ver_00859.csv")
OUT_PATH = Path("/home/bcrlab/igguest/porat_naama/data/processed/covid_clean.parquet")

df_loc_corona = pd.read_csv(RAW_PATH).copy()
(df_loc_corona == '<15').any().any()




True

In [23]:
df_loc_corona.replace('<15', 0, inplace=True)
print((df_loc_corona == '<15').any().any())
df_loc_corona = df_loc_corona.fillna(0)
df_loc_corona.head()

False


,town_code,agas_code,town,date,accumulated_cases,new_cases_on_date,accumulated_recoveries,new_recoveries_on_date,accumulated_hospitalized,new_hospitalized_on_date,accumulated_deaths,new_deaths_on_date,accumulated_diagnostic_tests,new_diagnostic_tests_on_date,accumulated_vaccination_first_dose,new_vacc_first_dose_on_date,accumulated_vaccination_second_dose,new_vacc_second_dose_on_date,accumulated_vaccination_third_dose,new_vacc_third_dose_on_date
0,26,0.0,ראש פינה,2020-03-11,0,False,0,False,0.0,False,0,False,0,False,0.0,False,0.0,False,0.0,False
1,26,0.0,ראש פינה,2020-03-12,0,False,0,False,0.0,False,0,False,0,False,0.0,False,0.0,False,0.0,False
2,26,0.0,ראש פינה,2020-03-13,0,False,0,False,0.0,False,0,False,0,False,0.0,False,0.0,False,0.0,False
3,26,0.0,ראש פינה,2020-03-14,0,False,0,False,0.0,False,0,False,0,False,0.0,False,0.0,False,0.0,False
4,26,0.0,ראש פינה,2020-03-15,0,False,0,False,0.0,False,0,False,0,False,0.0,False,0.0,False,0.0,False


In [24]:
# This cell removes agas_code = 0 for towns that also have specific non-zero statistical areas,
# keeps agas_code = 0 when it is the town's only statistical area,
# and creates a unique City_agas_code identifier (town_code + agas_code).

df_loc_corona['agas_code'] = df_loc_corona['agas_code'].astype(int)

# Identify towns that have at least one non-zero agas_code
towns_with_real_agas = df_loc_corona.loc[
    df_loc_corona['agas_code'] != 0,
    'town_code'
].unique()

# Drop agas_code == 0 only for towns that also have non-zero agas codes
df_loc_corona = df_loc_corona[
    ~(
        (df_loc_corona['agas_code'] == 0) &
        (df_loc_corona['town_code'].isin(towns_with_real_agas))
    )
].copy()

# Create combined city + statistical-area code
df_loc_corona['City_agas_code'] = (
    df_loc_corona['town_code'].astype(str)
    + "_"
    + df_loc_corona['agas_code'].astype(str)
)

df_loc_corona = df_loc_corona.set_index('City_agas_code')

df_loc_corona.head()

,town_code,agas_code,town,date,accumulated_cases,new_cases_on_date,accumulated_recoveries,new_recoveries_on_date,accumulated_hospitalized,new_hospitalized_on_date,accumulated_deaths,new_deaths_on_date,accumulated_diagnostic_tests,new_diagnostic_tests_on_date,accumulated_vaccination_first_dose,new_vacc_first_dose_on_date,accumulated_vaccination_second_dose,new_vacc_second_dose_on_date,accumulated_vaccination_third_dose,new_vacc_third_dose_on_date
City_agas_code,,,,,,,,,,,,,,,,,,,,
26_0,26,0,ראש פינה,2020-03-11,0,False,0,False,0.0,False,0,False,0,False,0.0,False,0.0,False,0.0,False
26_0,26,0,ראש פינה,2020-03-12,0,False,0,False,0.0,False,0,False,0,False,0.0,False,0.0,False,0.0,False
26_0,26,0,ראש פינה,2020-03-13,0,False,0,False,0.0,False,0,False,0,False,0.0,False,0.0,False,0.0,False
26_0,26,0,ראש פינה,2020-03-14,0,False,0,False,0.0,False,0,False,0,False,0.0,False,0.0,False,0.0,False
26_0,26,0,ראש פינה,2020-03-15,0,False,0,False,0.0,False,0,False,0,False,0.0,False,0.0,False,0.0,False


In [25]:
# This cell verifies that no town still contains both agas_code = 0 and non-zero agas codes.
# These should be empty: no town should have both x_0 and x_nonzero
check = (
    df_loc_corona.reset_index()
    .groupby('town_code')['agas_code']
    .agg(lambda x: (0 in x.values) and (x != 0).any())
)

print("Towns still containing both 0 and non-zero agas codes:")
print(check[check].index.tolist())

Towns still containing both 0 and non-zero agas codes:
[]


In [26]:
accumulated_cols = ['accumulated_cases', 'accumulated_recoveries', 'accumulated_hospitalized', 'accumulated_deaths', 'accumulated_diagnostic_tests', 'accumulated_vaccination_first_dose', 'accumulated_vaccination_second_dose', 'accumulated_vaccination_third_dose']
new_on_date_cols = ['new_cases_on_date', 'new_recoveries_on_date', 'new_hospitalized_on_date', 'new_deaths_on_date', 'new_diagnostic_tests_on_date', 'new_vacc_first_dose_on_date', 'new_vacc_second_dose_on_date', 'new_vacc_third_dose_on_date']

for col in accumulated_cols:
    df_loc_corona[col] = pd.to_numeric(df_loc_corona[col], errors='coerce').fillna(0).astype(int)

df_loc_corona = df_loc_corona.reset_index()
df_loc_corona['is_city_aggregate'] = 0

existing_city_codes = set(df_loc_corona['City_agas_code'])
towns_missing_city_row = sorted(
    t for t in df_loc_corona['town_code'].unique() if f"{t}_0" not in existing_city_codes
)

sub_rows = df_loc_corona[df_loc_corona['town_code'].isin(towns_missing_city_row)]
agg_dict = {**{c: 'sum' for c in accumulated_cols}, **{c: 'max' for c in new_on_date_cols}, 'town': 'first'}
city_agg = sub_rows.groupby(['town_code', 'date'], as_index=False).agg(agg_dict)
city_agg['agas_code'] = 0
city_agg['City_agas_code'] = city_agg['town_code'].astype(str) + "_0"
city_agg['is_city_aggregate'] = 1

df_loc_corona = pd.concat([df_loc_corona, city_agg], ignore_index=True)
df_loc_corona = df_loc_corona.sort_values(['City_agas_code', 'date']).set_index('City_agas_code')

print(f"Towns synthesized: {len(towns_missing_city_row)}")
print(f"Rows added: {len(city_agg)}")
print(f"New df.shape: {df_loc_corona.shape}")
print(f"Unique agas_code values: {sorted(df_loc_corona['agas_code'].unique())}")

Towns synthesized: 81
Rows added: 95094
New df.shape: (1942906, 21)
Unique agas_code values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 31, 32, 33, 34, 35, 36, 37, 38, 41, 42, 43, 44, 45, 51, 52, 53, 54, 55, 56, 57, 61, 62, 63, 64, 65, 66, 111, 112, 113, 114, 115, 116, 117, 121, 122, 123, 124, 125, 126, 127, 131, 132, 133, 134, 135, 136, 137, 138, 141, 142, 143, 144, 145, 211, 212, 213, 214, 215, 216, 217, 221, 222, 223, 224, 225, 226, 231, 232, 233, 234, 235, 236, 241, 242, 243, 244, 311, 312, 313, 314, 315, 316, 317, 321, 322, 323, 324, 325, 326, 331, 332, 333, 334, 335, 336, 341, 342, 343, 345, 346, 347, 351, 352, 354, 411, 412, 413, 414, 415, 416, 417, 418, 421, 422, 423, 424, 425, 426, 427, 431, 432, 433, 434, 435, 436, 437, 511, 512, 513, 514, 515, 516, 517, 521, 522, 523, 524, 525, 526, 527, 528, 531, 532, 533, 534, 541, 542, 544, 555, 611, 612, 613, 614, 615, 616, 617, 621, 622, 623, 624, 625, 626, 631, 632, 633, 641, 

In [27]:
df_loc_corona[df_loc_corona['date'] == '2022-03-16'].head()

,town_code,agas_code,town,date,accumulated_cases,new_cases_on_date,accumulated_recoveries,new_recoveries_on_date,accumulated_hospitalized,new_hospitalized_on_date,...,new_deaths_on_date,accumulated_diagnostic_tests,new_diagnostic_tests_on_date,accumulated_vaccination_first_dose,new_vacc_first_dose_on_date,accumulated_vaccination_second_dose,new_vacc_second_dose_on_date,accumulated_vaccination_third_dose,new_vacc_third_dose_on_date,is_city_aggregate
City_agas_code,,,,,,,,,,,,,,,,,,,,,
1015_0,1015,0,מבשרת ציון,2022-03-16,9840,True,9649,True,140,False,...,False,156915,True,19789,False,18693,True,15166,False,1
1015_1,1015,1,מבשרת ציון,2022-03-16,1136,True,1111,True,0,False,...,False,20535,True,2719,False,2620,False,2203,False,0
1015_2,1015,2,מבשרת ציון,2022-03-16,2652,True,2604,True,53,False,...,False,41070,True,5046,False,4728,False,3763,False,0
1015_3,1015,3,מבשרת ציון,2022-03-16,1498,True,1469,True,21,False,...,False,25539,True,3598,False,3448,False,2885,False,0
1015_4,1015,4,מבשרת ציון,2022-03-16,918,True,902,True,0,False,...,False,15640,True,1997,False,1905,True,1557,False,0


In [28]:
df_loc_corona = df_loc_corona.drop(columns=['new_cases_on_date', 'new_recoveries_on_date', 'new_hospitalized_on_date', 'new_deaths_on_date', 'new_diagnostic_tests_on_date', 'new_vacc_first_dose_on_date', 'new_vacc_second_dose_on_date', 'new_vacc_third_dose_on_date'])
df_loc_corona = df_loc_corona.drop(columns=['town_code','agas_code'])
df_loc_corona = df_loc_corona.sort_values(['City_agas_code', 'date'])
df_loc_corona.head()


,town,date,accumulated_cases,accumulated_recoveries,accumulated_hospitalized,accumulated_deaths,accumulated_diagnostic_tests,accumulated_vaccination_first_dose,accumulated_vaccination_second_dose,accumulated_vaccination_third_dose,is_city_aggregate
City_agas_code,,,,,,,,,,,
1015_0,מבשרת ציון,2020-03-11,0,0,0,0,0,0,0,0,1
1015_0,מבשרת ציון,2020-03-12,0,0,0,0,0,0,0,0,1
1015_0,מבשרת ציון,2020-03-13,0,0,0,0,0,0,0,0,1
1015_0,מבשרת ציון,2020-03-14,0,0,0,0,0,0,0,0,1
1015_0,מבשרת ציון,2020-03-15,0,0,0,0,0,0,0,0,1


In [29]:
columns_to_convert = ['accumulated_cases', 'accumulated_recoveries', 'accumulated_hospitalized', 'accumulated_deaths', 'accumulated_diagnostic_tests', 'accumulated_vaccination_first_dose', 'accumulated_vaccination_second_dose', 'accumulated_vaccination_third_dose']
for column in columns_to_convert:
    # Convert to numeric first, then cast to integer
    df_loc_corona[column] = pd.to_numeric(df_loc_corona[column], errors='coerce').fillna(0).astype(int)
df_loc_corona.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1942906 entries, 1015_0 to 99_0
Data columns (total 11 columns):
 #   Column                               Dtype 
---  ------                               ----- 
 0   town                                 object
 1   date                                 object
 2   accumulated_cases                    int64 
 3   accumulated_recoveries               int64 
 4   accumulated_hospitalized             int64 
 5   accumulated_deaths                   int64 
 6   accumulated_diagnostic_tests         int64 
 7   accumulated_vaccination_first_dose   int64 
 8   accumulated_vaccination_second_dose  int64 
 9   accumulated_vaccination_third_dose   int64 
 10  is_city_aggregate                    int64 
dtypes: int64(9), object(2)
memory usage: 177.9+ MB


In [30]:

df_loc_corona = df_loc_corona.copy()

# (also important: sort before diff)
df_loc_corona = df_loc_corona.sort_values(["City_agas_code", "date"])

df_loc_corona["positive_daily"]      = df_loc_corona.groupby("City_agas_code")["accumulated_cases"].diff()
df_loc_corona["hospitalized_daily"]  = df_loc_corona.groupby("City_agas_code")["accumulated_hospitalized"].diff()
df_loc_corona["tested_daily"]        = df_loc_corona.groupby("City_agas_code")["accumulated_diagnostic_tests"].diff()

df_loc_corona["infection_rate"]   = df_loc_corona["positive_daily"] / df_loc_corona["tested_daily"]
df_loc_corona["recovery_ratio"]   = df_loc_corona["accumulated_recoveries"] / df_loc_corona["accumulated_cases"]
df_loc_corona["recovery_index"]   = df_loc_corona["accumulated_recoveries"] / df_loc_corona["accumulated_diagnostic_tests"]

# Replace inf/-inf -> NaN, then decide how to fill
df_loc_corona["infection_rate"] = df_loc_corona["infection_rate"].replace([np.inf, -np.inf], np.nan)

# Now choose fill policy
df_loc_corona = df_loc_corona.fillna(0)
df_loc_corona.tail()

,town,date,accumulated_cases,accumulated_recoveries,accumulated_hospitalized,accumulated_deaths,accumulated_diagnostic_tests,accumulated_vaccination_first_dose,accumulated_vaccination_second_dose,accumulated_vaccination_third_dose,is_city_aggregate,positive_daily,hospitalized_daily,tested_daily,infection_rate,recovery_ratio,recovery_index
City_agas_code,,,,,,,,,,,,,,,,,
99_0,מצפה רמון,2023-05-24,2676,2666,34,0,32481,3545,3323,2521,0,0.0,0.0,0.0,0.0,0.996263,0.082079
99_0,מצפה רמון,2023-05-25,2676,2666,34,0,32481,3545,3323,2521,0,0.0,0.0,0.0,0.0,0.996263,0.082079
99_0,מצפה רמון,2023-05-26,2676,2666,34,0,32481,3545,3323,2521,0,0.0,0.0,0.0,0.0,0.996263,0.082079
99_0,מצפה רמון,2023-05-27,2676,2666,34,0,32482,3545,3323,2521,0,0.0,0.0,1.0,0.0,0.996263,0.082076
99_0,מצפה רמון,2023-05-28,2676,2666,34,0,32482,3545,3323,2521,0,0.0,0.0,0.0,0.0,0.996263,0.082076


In [31]:
# ============================================================
# Adaptive infection rate
# If too few tests were performed on a single day,
# expand the window backward until enough tests accumulate.
# ============================================================

MIN_TESTS = 30
MAX_WINDOW_DAYS = 14

# Keep the original 1-day calculation for comparison/debugging
df_loc_corona["infection_rate_daily_raw"] = df_loc_corona["infection_rate"]

# Final adaptive values
df_loc_corona["infection_rate_adaptive"] = np.nan
df_loc_corona["infection_rate_window_days"] = np.nan
df_loc_corona["infection_rate_tests_used"] = np.nan
df_loc_corona["infection_rate_cases_used"] = np.nan

# Make sure date is datetime
df_loc_corona["date"] = pd.to_datetime(df_loc_corona["date"])

grouped = df_loc_corona.groupby("City_agas_code", sort=False)

# Try windows of 1 day, 2 days, ..., up to MAX_WINDOW_DAYS
for lag in range(1, MAX_WINDOW_DAYS + 1):

    prev_cases = grouped["accumulated_cases"].shift(lag)
    prev_tests = grouped["accumulated_diagnostic_tests"].shift(lag)
    prev_date = grouped["date"].shift(lag)

    cases_in_window = (
        df_loc_corona["accumulated_cases"] - prev_cases
    )

    tests_in_window = (
        df_loc_corona["accumulated_diagnostic_tests"] - prev_tests
    )

    actual_window_days = (
        df_loc_corona["date"] - prev_date
    ).dt.days

    # Only assign rows that have not already found a suitable window
    not_assigned = df_loc_corona["infection_rate_adaptive"].isna()

    # A usable window must:
    # 1. contain at least MIN_TESTS tests
    # 2. have non-negative case/test differences
    # 3. not imply more cases than tests
    # 4. stay within MAX_WINDOW_DAYS
    valid = (
        not_assigned
        & (tests_in_window >= MIN_TESTS)
        & (cases_in_window >= 0)
        & (cases_in_window <= tests_in_window)
        & (actual_window_days >= 1)
        & (actual_window_days <= MAX_WINDOW_DAYS)
    )

    df_loc_corona.loc[valid, "infection_rate_adaptive"] = (
        cases_in_window[valid] / tests_in_window[valid]
    )

    df_loc_corona.loc[valid, "infection_rate_window_days"] = (
        actual_window_days[valid]
    )

    df_loc_corona.loc[valid, "infection_rate_tests_used"] = (
        tests_in_window[valid]
    )

    df_loc_corona.loc[valid, "infection_rate_cases_used"] = (
        cases_in_window[valid]
    )


# Replace the old infection_rate with the adaptive version
df_loc_corona["infection_rate"] = df_loc_corona["infection_rate_adaptive"]


# ============================================================
# Diagnostics
# ============================================================

print("Adaptive infection-rate calculation")
print("-----------------------------------")
print(f"Minimum tests required: {MIN_TESTS}")
print(f"Maximum window:         {MAX_WINDOW_DAYS} days")

print(
    "\nRows with a usable infection rate:",
    f"{df_loc_corona['infection_rate'].notna().sum():,}",
    f"({df_loc_corona['infection_rate'].notna().mean()*100:.2f}%)"
)

print(
    "Rows still unresolved:",
    f"{df_loc_corona['infection_rate'].isna().sum():,}",
    f"({df_loc_corona['infection_rate'].isna().mean()*100:.2f}%)"
)

print(
    "Rows with infection_rate > 1:",
    f"{(df_loc_corona['infection_rate'] > 1).sum():,}"
)

print("\nWindow lengths used:")
print(
    df_loc_corona["infection_rate_window_days"]
    .value_counts(dropna=False)
    .sort_index()
)

Adaptive infection-rate calculation
-----------------------------------
Minimum tests required: 30
Maximum window:         14 days

Rows with a usable infection rate: 1,660,817 (85.48%)
Rows still unresolved: 282,089 (14.52%)
Rows with infection_rate > 1: 0

Window lengths used:
1.0     509558
2.0     346097
3.0     202508
4.0     134088
5.0      95875
6.0      74331
7.0      60138
8.0      49769
9.0      43333
10.0     37734
11.0     32804
12.0     28617
13.0     24580
14.0     21385
NaN     282089
Name: infection_rate_window_days, dtype: int64


In [32]:
df_loc_corona['is_lockdown'] = 0
df_loc_corona['date']= pd.to_datetime(df_loc_corona['date'])
df_loc_corona.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1942906 entries, 1015_0 to 99_0
Data columns (total 23 columns):
 #   Column                               Dtype         
---  ------                               -----         
 0   town                                 object        
 1   date                                 datetime64[ns]
 2   accumulated_cases                    int64         
 3   accumulated_recoveries               int64         
 4   accumulated_hospitalized             int64         
 5   accumulated_deaths                   int64         
 6   accumulated_diagnostic_tests         int64         
 7   accumulated_vaccination_first_dose   int64         
 8   accumulated_vaccination_second_dose  int64         
 9   accumulated_vaccination_third_dose   int64         
 10  is_city_aggregate                    int64         
 11  positive_daily                       float64       
 12  hospitalized_daily                   float64       
 13  tested_daily                  

In [33]:
#lockdown dates:
#March 19, 2020 – April 24, 2020
#September 18, 2020 – October 18, 2020
#December 27, 2020 – February 6, 2021
#July 29, 2021 – November 27, 2021

lockdown_periods = [('2020-03-19', '2020-04-24'), ('2020-09-18', '2020-10-18'), ('2020-12-27', '2021-02-06'), ('2021-07-29', '2021-11-27')]

lockdown_periods = [(pd.Timestamp(start), pd.Timestamp(end)) for start, end in lockdown_periods]


for start_date, end_date in lockdown_periods:
    mask = (df_loc_corona['date'] >= start_date) & (df_loc_corona['date'] <= end_date)
    df_loc_corona.loc[mask, 'is_lockdown'] = 1

df_loc_corona.tail()

,town,date,accumulated_cases,accumulated_recoveries,accumulated_hospitalized,accumulated_deaths,accumulated_diagnostic_tests,accumulated_vaccination_first_dose,accumulated_vaccination_second_dose,accumulated_vaccination_third_dose,...,tested_daily,infection_rate,recovery_ratio,recovery_index,infection_rate_daily_raw,infection_rate_adaptive,infection_rate_window_days,infection_rate_tests_used,infection_rate_cases_used,is_lockdown
City_agas_code,,,,,,,,,,,,,,,,,,,,,
99_0,מצפה רמון,2023-05-24,2676,2666,34,0,32481,3545,3323,2521,...,0.0,NaN,0.996263,0.082079,0.0,NaN,NaN,NaN,NaN,0
99_0,מצפה רמון,2023-05-25,2676,2666,34,0,32481,3545,3323,2521,...,0.0,NaN,0.996263,0.082079,0.0,NaN,NaN,NaN,NaN,0
99_0,מצפה רמון,2023-05-26,2676,2666,34,0,32481,3545,3323,2521,...,0.0,NaN,0.996263,0.082079,0.0,NaN,NaN,NaN,NaN,0
99_0,מצפה רמון,2023-05-27,2676,2666,34,0,32482,3545,3323,2521,...,1.0,NaN,0.996263,0.082076,0.0,NaN,NaN,NaN,NaN,0
99_0,מצפה רמון,2023-05-28,2676,2666,34,0,32482,3545,3323,2521,...,0.0,NaN,0.996263,0.082076,0.0,NaN,NaN,NaN,NaN,0


In [34]:
df_loc_corona[df_loc_corona['date']=='2020-03-16'].head()

,town,date,accumulated_cases,accumulated_recoveries,accumulated_hospitalized,accumulated_deaths,accumulated_diagnostic_tests,accumulated_vaccination_first_dose,accumulated_vaccination_second_dose,accumulated_vaccination_third_dose,...,tested_daily,infection_rate,recovery_ratio,recovery_index,infection_rate_daily_raw,infection_rate_adaptive,infection_rate_window_days,infection_rate_tests_used,infection_rate_cases_used,is_lockdown
City_agas_code,,,,,,,,,,,,,,,,,,,,,
1015_0,מבשרת ציון,2020-03-16,0,0,0,0,0,0,0,0,...,0.0,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0
1015_1,מבשרת ציון,2020-03-16,0,0,0,0,0,0,0,0,...,0.0,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0
1015_2,מבשרת ציון,2020-03-16,0,0,0,0,0,0,0,0,...,0.0,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0
1015_3,מבשרת ציון,2020-03-16,0,0,0,0,0,0,0,0,...,0.0,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0
1015_4,מבשרת ציון,2020-03-16,0,0,0,0,0,0,0,0,...,0.0,NaN,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0


In [35]:
if df_loc_corona.index.name == "City_agas_code":
    df_loc_corona = df_loc_corona.reset_index()
df_loc_corona = df_loc_corona.sort_values(["City_agas_code", "date"]).reset_index(drop=True)
g = df_loc_corona.groupby("City_agas_code")

for c in ["infection_rate"]:
    for k in [1, 2, 3, 7, 14]:
        df_loc_corona[f"{c}_lag{k}"] = g[c].shift(k)
    for w in [7, 14, 28]:
        df_loc_corona[f"{c}_roll{w}"] = (
            g[c].rolling(w, min_periods=w).mean().reset_index(level=0, drop=True)
        )



print(df_loc_corona.shape)



(1942906, 32)


In [36]:
df_loc_corona.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1942906 entries, 0 to 1942905
Data columns (total 32 columns):
 #   Column                               Dtype         
---  ------                               -----         
 0   City_agas_code                       object        
 1   town                                 object        
 2   date                                 datetime64[ns]
 3   accumulated_cases                    int64         
 4   accumulated_recoveries               int64         
 5   accumulated_hospitalized             int64         
 6   accumulated_deaths                   int64         
 7   accumulated_diagnostic_tests         int64         
 8   accumulated_vaccination_first_dose   int64         
 9   accumulated_vaccination_second_dose  int64         
 10  accumulated_vaccination_third_dose   int64         
 11  is_city_aggregate                    int64         
 12  positive_daily                       float64       
 13  hospitalized_daily         

In [37]:
df_loc_corona = df_loc_corona.drop(columns=['accumulated_cases','accumulated_hospitalized','accumulated_diagnostic_tests','accumulated_recoveries','accumulated_deaths','positive_daily', 'hospitalized_daily'])

In [38]:
df_loc_corona = df_loc_corona.fillna(0)
df_loc_corona.tail()

,City_agas_code,town,date,accumulated_vaccination_first_dose,accumulated_vaccination_second_dose,accumulated_vaccination_third_dose,is_city_aggregate,tested_daily,infection_rate,recovery_ratio,...,infection_rate_cases_used,is_lockdown,infection_rate_lag1,infection_rate_lag2,infection_rate_lag3,infection_rate_lag7,infection_rate_lag14,infection_rate_roll7,infection_rate_roll14,infection_rate_roll28
1942901,99_0,מצפה רמון,2023-05-24,3545,3323,2521,0,0.0,0.0,0.996263,...,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1942902,99_0,מצפה רמון,2023-05-25,3545,3323,2521,0,0.0,0.0,0.996263,...,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1942903,99_0,מצפה רמון,2023-05-26,3545,3323,2521,0,0.0,0.0,0.996263,...,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1942904,99_0,מצפה רמון,2023-05-27,3545,3323,2521,0,1.0,0.0,0.996263,...,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1942905,99_0,מצפה רמון,2023-05-28,3545,3323,2521,0,0.0,0.0,0.996263,...,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [39]:
df_loc_corona[df_loc_corona['date'] == '2022-03-16'].head()

,City_agas_code,town,date,accumulated_vaccination_first_dose,accumulated_vaccination_second_dose,accumulated_vaccination_third_dose,is_city_aggregate,tested_daily,infection_rate,recovery_ratio,...,infection_rate_cases_used,is_lockdown,infection_rate_lag1,infection_rate_lag2,infection_rate_lag3,infection_rate_lag7,infection_rate_lag14,infection_rate_roll7,infection_rate_roll14,infection_rate_roll28
735,1015_0,מבשרת ציון,2022-03-16,19789,18693,15166,1,134.0,0.179104,0.980589,...,24.0,0,0.146199,0.160428,0.046256,0.212389,0.181818,0.157491,0.152650,0.161210
1909,1015_1,מבשרת ציון,2022-03-16,2719,2620,2203,0,18.0,0.219512,0.977993,...,9.0,0,0.177778,0.060606,0.025974,0.219512,0.195652,0.155976,0.173383,0.165749
3083,1015_2,מבשרת ציון,2022-03-16,5046,4728,3763,0,27.0,0.149425,0.981900,...,13.0,0,0.150000,0.212766,0.007874,0.297297,0.238095,0.150633,0.144122,0.151911
4257,1015_3,מבשרת ציון,2022-03-16,3598,3448,2885,0,31.0,0.129032,0.980641,...,4.0,0,0.132075,0.093023,0.066667,0.109091,0.178571,0.120345,0.128664,0.156172
5431,1015_4,מבשרת ציון,2022-03-16,1997,1905,1557,0,16.0,0.093750,0.982571,...,3.0,0,0.090909,0.086957,0.076923,0.222222,0.028571,0.110199,0.100126,0.114467


In [40]:
# ============================================================
# Clean up helper columns from adaptive infection-rate fix
# ============================================================

cols_to_drop = [
    "infection_rate_daily_raw",      # old problematic daily calculation
    "infection_rate_adaptive",       # duplicate of final infection_rate
    "infection_rate_tests_used",     # only needed for diagnostics
    "infection_rate_cases_used",     # only needed for diagnostics
]

df_loc_corona = df_loc_corona.drop(
    columns=cols_to_drop,
    errors="ignore"
)

df_loc_corona.head()

,City_agas_code,town,date,accumulated_vaccination_first_dose,accumulated_vaccination_second_dose,accumulated_vaccination_third_dose,is_city_aggregate,tested_daily,infection_rate,recovery_ratio,...,infection_rate_window_days,is_lockdown,infection_rate_lag1,infection_rate_lag2,infection_rate_lag3,infection_rate_lag7,infection_rate_lag14,infection_rate_roll7,infection_rate_roll14,infection_rate_roll28
0,1015_0,מבשרת ציון,2020-03-11,0,0,0,1,0.0,0.0,0.0,...,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1015_0,מבשרת ציון,2020-03-12,0,0,0,1,0.0,0.0,0.0,...,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1015_0,מבשרת ציון,2020-03-13,0,0,0,1,0.0,0.0,0.0,...,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1015_0,מבשרת ציון,2020-03-14,0,0,0,1,0.0,0.0,0.0,...,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1015_0,מבשרת ציון,2020-03-15,0,0,0,1,0.0,0.0,0.0,...,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [41]:
df_loc_corona.to_csv("/home/bcrlab/igguest/porat_naama/data/processed/Corona_loc_processed.csv")

print("Saved:", "/home/bcrlab/igguest/porat_naama/data/processed/Corona_loc_processed.csv")

Saved: /home/bcrlab/igguest/porat_naama/data/processed/Corona_loc_processed.csv
